# Imports

In [1]:
import os, gc, json, pickle
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    AutoModel, CLIPTextModel
)

from captum.attr import IntegratedGradients, Saliency
from scipy.stats import spearmanr
import torch.nn.functional as F


/home/aysel/tfe/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
CURRENT_DATASET = "Flickr8k"
BASE_DIR = "TFE_Data"
DATASETS_DIR = os.path.join(BASE_DIR, "Datasets")

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Using device:", device)


Using device: cuda


In [3]:
# Data Loader
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, max_len=64):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        return enc


# Load Models

In [4]:
class TextWrapper(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, input_ids, attention_mask=None):
        out = self.backbone(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True
        )
        cls = out.last_hidden_state[:, 0, :]
        return self.head(cls)

    def get_input_embeddings(self):
        return self.backbone.get_input_embeddings()


In [5]:
def load_bert(device):
    tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    model = AutoModelForSequenceClassification.from_pretrained(
        "bert-base-uncased",
        num_labels=2
    ).to(device).eval()
    backbone = model.bert
    return model, tok, backbone


def load_roberta(device):
    tok = AutoTokenizer.from_pretrained("roberta-base")
    model = AutoModelForSequenceClassification.from_pretrained(
        "roberta-base",
        num_labels=2
    ).to(device).eval()
    return model, tok, model.roberta

def load_gpt2(device):
    tok = AutoTokenizer.from_pretrained("gpt2")
    tok.pad_token = tok.eos_token  # OBLIGATOIRE
    model = AutoModelForSequenceClassification.from_pretrained(
        "gpt2",
        num_labels=2,
        pad_token_id=tok.pad_token_id  # OBLIGATOIRE
    ).to(device).eval()
    return model, tok, model.transformer


def load_clip_text(device):
    tok = AutoTokenizer.from_pretrained("openai/clip-vit-base-patch32")
    tok.pad_token = tok.eos_token  # indispensable

    backbone = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32")
    head = nn.Linear(backbone.config.hidden_size, 2)

    model = TextWrapper(backbone, head).to(device).eval()
    return model, tok, backbone


# Captum Attributions

In [6]:
class EmbeddingWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.embeddings = model.get_input_embeddings()

    def forward(self, embeddings, attention_mask=None):
        outputs = self.model(
            inputs_embeds=embeddings,
            attention_mask=attention_mask
        )
        return outputs.logits if hasattr(outputs, "logits") else outputs



In [7]:
class CLIPIGWrapper(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head = head

    def forward(self, embeddings, attention_mask=None):
        out = self.backbone(
            inputs_embeds=embeddings,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True
        )
        cls = out.last_hidden_state[:, 0, :]
        return self.head(cls)


In [8]:
def explain_ig(model, batch, target):
    emb_layer = model.get_input_embeddings()
    input_embeds = emb_layer(batch["input_ids"])

    # Si modèle = CLIP-Text → wrapper IG spécial
    if isinstance(model, TextWrapper):
        wrapper = CLIPIGWrapper(model.backbone, model.head)
    else:
        wrapper = EmbeddingWrapper(model)

    ig = IntegratedGradients(wrapper)

    attributions = ig.attribute(
        input_embeds,
        additional_forward_args=(batch["attention_mask"],),
        target=target
    )

    return attributions.norm(dim=-1)


In [9]:
def explain_saliency(model, batch, target):
    model.zero_grad()
    input_ids = batch["input_ids"]
    attention_mask = batch["attention_mask"]

    input_ids = input_ids.clone().detach().requires_grad_(True)

    out = model(input_ids, attention_mask=attention_mask)
    logits = out.logits if hasattr(out, "logits") else out
    loss = logits[0, target]
    loss.backward()

    return input_ids.grad.abs()


In [10]:
models_text = {
    "BERT": load_bert(device),
    "RoBERTa": load_roberta(device),
    "GPT-2": load_gpt2(device),
}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
all_attrs_text = []



df_text = pd.read_pickle(os.path.join(DATASETS_DIR, "df_Flickr8k.pkl"))

# IMPORTANT: Flickr8k has 5 captions per image → we take only the first
TEXTS = [caps[0] for caps in df_text["captions"].tolist()]
TEXTS = TEXTS[:50]

for name, (model, tok, backbone) in models_text.items():
    dataset = TextDataset(TEXTS, tok)
    loader = torch.utils.data.DataLoader(dataset, batch_size=1)

    model_attrs = []

    for idx, batch in enumerate(loader):
        batch = {
            "input_ids": batch["input_ids"].squeeze(0).long().to(device),
            "attention_mask": batch["attention_mask"].squeeze(0).long().to(device)
        }
        
        out = model(batch["input_ids"], attention_mask=batch["attention_mask"])
        logits = out.logits if hasattr(out, "logits") else out
        label = logits.argmax(dim=1).item()

        if name == "CLIP-Text":
            ig_attr = explain_saliency(model, batch, label)
        else:
            ig_attr = explain_ig(model, batch, label)

        model_attrs.append({
            "text_idx": idx,
            "label": label,
            "IG": ig_attr.cpu(),
            "input_ids": batch["input_ids"].cpu()
        })

    all_attrs_text.append({"model": name, "attrs": model_attrs})

with open("captum_text.pkl", "wb") as f:
    pickle.dump(all_attrs_text, f)


[transformers] GPT2ForSequenceClassification will not detect padding tokens in `inputs_embeds`. Results may be unexpected if using padding tokens in conjunction with `inputs_embeds.`


# Metrics

In [29]:
def faithfulness_text(model, batch, label, attr):
    model_cpu = model.to("cpu").eval()
    ids = batch["input_ids"].clone().cpu()

    base = model_cpu(ids, attention_mask=batch["attention_mask"].cpu()).logits.softmax(1)[0, label].item()

    impacts = []
    attrs = attr.squeeze().detach().cpu().numpy()
    for i in range(len(ids[0])):
        ids_mod = ids.clone()
        ids_mod[0, i] = 0
        out = model_cpu(ids_mod, attention_mask=batch["attention_mask"].cpu()).logits.softmax(1)[0, label].item()
        impacts.append(base - out)

    corr, _ = spearmanr(attrs, impacts)
    return corr


In [30]:
def rankcorr_text(model, batch, label, attr):
    return faithfulness_text(model, batch, label, attr)


In [31]:
def entropy_text(attr):
    a = attr.squeeze().detach().cpu().numpy()
    a = np.abs(a)
    a = a / (a.sum() + 1e-8)
    return -np.sum(a * np.log(a + 1e-8))


In [32]:
def attention_alignment(model, backbone, batch, attr):
    with torch.no_grad():
        out = backbone(
            batch["input_ids"].cpu(),
            attention_mask=batch["attention_mask"].cpu(),
            output_attentions=True,
            return_dict=True
        )
        attn = out.attentions[-1].mean(1).mean(1).cpu().numpy()

    a = attr.squeeze().detach().cpu().numpy()
    corr, _ = spearmanr(a, attn)
    return corr


In [35]:
def stability_text(model, batch, label, attr):
    model_cpu = model.to("cpu").eval()

    # 1. Embeddings
    emb_layer = model_cpu.get_input_embeddings()
    input_embeds = emb_layer(batch["input_ids"].cpu())

    # 2. Wrapper IG
    wrapper = EmbeddingWrapper(model_cpu)
    ig = IntegratedGradients(wrapper)

    def aug_embeds(embeds):
        embeds = embeds.clone()
        if embeds.shape[1] > 3:
            embeds[0,1] = embeds[0,2]
        return embeds

    sims = []
    for _ in range(1):
        embeds_aug = aug_embeds(input_embeds)

        attr_aug = ig.attribute(
            embeds_aug,
            additional_forward_args=(batch["attention_mask"].cpu(),),
            target=label
        )

        # 🔥 Réduction identique à explain_ig
        attr_aug = attr_aug.norm(dim=-1)

        a1 = attr.detach().cpu().numpy().flatten()
        a2 = attr_aug.detach().cpu().numpy().flatten()

        sim = np.dot(a1, a2) / (np.linalg.norm(a1)*np.linalg.norm(a2) + 1e-8)
        sims.append(sim)

    return float(np.mean(sims))


In [36]:
text_results = []

for entry in all_attrs_text:
    model_name = entry["model"]
    model, tok, backbone = models_text[model_name]
    print(f"Evaluating {model_name}...")

    model_scores = []

    for item in entry["attrs"]:
        idx = item["text_idx"]
        label = item["label"]
        attr = item["IG"]
        ids = item["input_ids"]

        batch = {"input_ids": ids, "attention_mask": (ids!=0).long()}

        scores = {
            "FaithfulnessCorrelation": faithfulness_text(model, batch, label, attr),
            "RankCorrelation": rankcorr_text(model, batch, label, attr),
            "Entropy": entropy_text(attr),
            "Stability": stability_text(model, batch, label, attr)
        }

        model_scores.append(scores)

    text_results.append({"model": model_name, "scores": model_scores})


Evaluating BERT...
Evaluating RoBERTa...
Evaluating GPT-2...


In [37]:
rows = []

for entry in text_results:
    model = entry["model"]
    for s in entry["scores"]:
        for metric, value in s.items():
            rows.append([model, metric, float(value)])

df_text = pd.DataFrame(rows, columns=["model","metric","value"])
summary_text = df_text.groupby(["model","metric"]).mean().reset_index()
summary_text


,model,metric,value
0,BERT,Entropy,2.561380
1,BERT,FaithfulnessCorrelation,0.013782
2,BERT,RankCorrelation,0.013782
3,BERT,Stability,0.973402
4,GPT-2,Entropy,2.311935
5,GPT-2,FaithfulnessCorrelation,-0.133464
6,GPT-2,RankCorrelation,-0.133464
7,GPT-2,Stability,0.851668
8,RoBERTa,Entropy,2.524101
9,RoBERTa,FaithfulnessCorrelation,0.232595
